# 🍏 Apple Generative Imagery Systems - FLUX.1 [dev] LoRA Training Pipeline
### Target: FLUX.1-dev (12B MMDiT) | Dataset: 21 Curated Apple Minimal LookDev Assets
---
이 주피터 노트북은 Google Colab(T4 16GB / L4 24GB / A100) 환경에서 **Apple 스탠다드 미니멀 미학(무광 아노다이징 알루미늄, CMF 디테일, 앰비언트 오클루전 라이팅)**을 가진 **FLUX.1-dev LoRA**를 학습하는 올인원 자동화 파이프라인입니다.

**핵심 기술 사양:**
- **엔진**: Ostris AI-Toolkit (FLUX 공식 표준 LoRA 학습 프레임워크)
- **아키텍처**: 12B Flow Matching Transformer (MMDiT)
- **최적화**: FP8 Base Model Caching + 8-bit AdamW Optimizer + Gradient Checkpointing (Colab T4 16GB 완벽 구동 지원)
- **트리거 토큰**: `apple minimal craft style`, `clean matte studio lookdev`

In [ ]:
# 1. GPU 하드웨어 가속기 사양 확인 (Colab 런타임 유형이 GPU로 되어 있는지 확인)
!nvidia-smi

In [ ]:
# 2. Google Drive 마운트
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3. FLUX 전용 고속 학습 툴킷(AI-Toolkit) 복제 및 의존성 설치
%cd /content
!git clone https://github.com/ostris/ai-toolkit.git /content/ai-toolkit
%cd /content/ai-toolkit
!git submodule update --init --recursive

# 필수 패키지 설치
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install -q -r requirements.txt
!pip install -q bitsandbytes accelerate huggingface_hub

In [ ]:
# 4. Hugging Face 로그인 (FLUX.1-dev 모델 가중치 접근용 Token 입력)
# https://huggingface.co/settings/tokens 에서 발급받은 Access Token을 입력하세요.
from huggingface_hub import login
import getpass

print('🔑 Hugging Face Access Token을 입력하세요 (FLUX.1-dev 동의 완료된 계정):')
hf_token = getpass.getpass('HF Token: ')
login(token=hf_token)
print('✅ Hugging Face 로그인 완료!')

In [ ]:
# 5. 데이터셋(21장 이미지 + 캡션) 로드 및 배치
import os, shutil, zipfile

DATASET_DIR = '/content/dataset'
os.makedirs(DATASET_DIR, exist_ok=True)

# 1) Google Drive에서 검색
candidate_paths = [
    '/content/drive/MyDrive/Generative-Imagery-Systems/apple_lora_project/cropped_1024',
    '/content/drive/MyDrive/spatial-gen-pipeline/dataset/cropped_1024',
    '/content/drive/MyDrive/cropped_1024',
    '/content/drive/MyDrive/cropped_1024_dataset.zip',
    '/content/drive/MyDrive/spatial-gen-pipeline/dataset/cropped_1024_dataset.zip'
]

found_source = None
for path in candidate_paths:
    if os.path.exists(path):
        found_source = path
        break

# 2) GitHub 레포지토리에서 직접 복제 (드라이브에 없을 경우 자동 Fallback)
if not found_source:
    print('🌐 GitHub에서 최신 21장 큐레이션 데이터셋 직접 다운로드 중...')
    !git clone https://github.com/djbyun/spatial-gen-pipeline.git /content/repo_temp
    repo_data_dir = '/content/repo_temp/dataset/cropped_1024'
    if os.path.exists(repo_data_dir):
        found_source = repo_data_dir

if found_source:
    print(f'📁 데이터셋 소스: {found_source}')
    if found_source.endswith('.zip'):
        with zipfile.ZipFile(found_source, 'r') as zip_ref:
            zip_ref.extractall(DATASET_DIR)
    else:
        for f in os.listdir(found_source):
            if f.endswith('.png') or f.endswith('.txt'):
                shutil.copy2(os.path.join(found_source, f), os.path.join(DATASET_DIR, f))
    
    img_files = [f for f in os.listdir(DATASET_DIR) if f.endswith('.png')]
    txt_files = [f for f in os.listdir(DATASET_DIR) if f.endswith('.txt')]
    print(f'✅ 데이터셋 준비 완료: 이미지 {len(img_files)}장 + 캡션 {len(txt_files)}개')
else:
    print('⚠️ 데이터셋을 찾을 수 없습니다. 왼쪽 파일 창 /content/dataset 에 직접 업로드해 주세요.')

In [ ]:
# 6. FLUX LoRA 학습용 최적화 설정 파일 (YAML) 생성
config_yaml_content = """
job: extension
config:
  name: "apple_minimal_craft_flux_v1"
  process:
    - type: 'sd_trainer'
      training_folder: "/content/output_flux_lora"
      device: cuda:0
      trigger_word: "apple minimal craft style"
      network:
        type: "lora"
        linear: 16
        linear_alpha: 16
      save:
        dtype: float16
        save_every: 250
        max_step_saves_to_keep: 4
      datasets:
        - folder_path: "/content/dataset"
          caption_ext: "txt"
          caption_dropout_rate: 0.05
          shuffle_tokens: false
          cache_latents_to_disk: true
          resolution: [1024]
      train:
        batch_size: 1
        steps: 1200
        gradient_accumulation_steps: 1
        train_unet: true
        train_text_encoder: false
        gradient_checkpointing: true
        noise_scheduler: "flowmatch"
        optimizer: "adamw8bit"
        lr: 1e-4
        ema_config:
          use_ema: false
        dtype: bf16
      model:
        name_or_path: "black-forest-labs/FLUX.1-dev"
        is_flux: true
        quantize: true
        low_vram: true
      sample:
        sampler: "flowmatch"
        sample_every: 250
        width: 1024
        height: 1024
        prompts:
          - "apple minimal craft style, clean matte studio lookdev, anodized aluminum cylinder on neutral white podium, soft studio lighting"
          - "apple minimal craft style, organic curved frosted glass object, ambient occlusion lighting, neutral background"
        neg: ""
        seed: 42
        walk_seed: true
        guidance_scale: 3.5
        sample_steps: 25
"""

os.makedirs('/content/ai-toolkit/config', exist_ok=True)
with open('/content/ai-toolkit/config/apple_craft_flux.yaml', 'w') as f:
    f.write(config_yaml_content.strip())

print('✅ FLUX LoRA 학습 설정 파일(/content/ai-toolkit/config/apple_craft_flux.yaml) 생성 완료!')

In [ ]:
# 7. FLUX.1-dev LoRA 파인튜닝 실행 (약 15~25분 소요)
%cd /content/ai-toolkit
!python run.py config/apple_craft_flux.yaml

In [ ]:
# 8. 완성된 LoRA 가중치(.safetensors)를 Google Drive 및 로컬로 백업
import os, glob
from google.colab import files

DEST_DIR = '/content/drive/MyDrive/spatial-gen-pipeline/weights'
os.makedirs(DEST_DIR, exist_ok=True)

saved_loras = glob.glob('/content/output_flux_lora/**/*.safetensors', recursive=True)
if saved_loras:
    for lora_path in saved_loras:
        fname = os.path.basename(lora_path)
        shutil.copy2(lora_path, os.path.join(DEST_DIR, fname))
        print(f'✅ Google Drive 백업 완료: {fname}')
    
    # 최종 체크포인트 다운로드 트리거
    latest_lora = sorted(saved_loras)[-1]
    print(f'📥 최신 LoRA 파일 다운로드 시작: {latest_lora}')
    files.download(latest_lora)
else:
    print('⚠️ 저장된 LoRA 가중치 파일을 찾지 못했습니다.')